# Task 4 Ensemble Learning

**Objective:** Add cluster information to ensemble regressors, compare against the Task 2 baseline, and export the final results summary.

**Inputs:** `data/clustered.csv`, `models/supervised_best.pkl`

**Outputs:** feature importance, learning curve, and `reports/final_results_summary.csv`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'
RANDOM_STATE = 42
EPS = 1e-6
def savefig(filename):
    plt.tight_layout(); plt.savefig(REPORTS_DIR / filename, dpi=300, bbox_inches='tight'); plt.show()

In [2]:
from math import sqrt
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
df = pd.read_csv(DATA_DIR / 'clustered.csv')
df['cluster_label'] = df['cluster_label'].astype(str)
df['sulfur_ratio'] = df['free sulfur dioxide'] / (df['total sulfur dioxide'] + EPS)
df['alcohol_density_ratio'] = df['alcohol'] / (df['density'] + EPS)
df['acidity_balance'] = df['fixed acidity'] / (df['volatile acidity'] + df['citric acid'] + EPS)
feature_cols=[c for c in df.columns if c!='quality']
X=df[feature_cols]; y=df['quality']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=RANDOM_STATE)
pre = ColumnTransformer([
 ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), [c for c in feature_cols if c!='cluster_label']),
 ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), ['cluster_label'])
])
models = {'Random Forest': Pipeline([('preprocessor', pre), ('model', RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=1))]), 'Gradient Boosting': Pipeline([('preprocessor', pre), ('model', GradientBoostingRegressor(n_estimators=180, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))])}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
baseline = joblib.load(MODELS_DIR / 'supervised_best.pkl')
rows=[]
baseline_pred = baseline['pipeline'].predict(X_test[baseline['feature_columns']])
rows.append({'Model_Type':'Task 2 Baseline','Model':baseline['model_name'],'CV_RMSE':np.nan,'CV_MAE':np.nan,'CV_R2':np.nan,'Test_RMSE':sqrt(mean_squared_error(y_test, baseline_pred)),'Test_MAE':mean_absolute_error(y_test, baseline_pred),'Test_R2':r2_score(y_test, baseline_pred)})
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=('neg_root_mean_squared_error','neg_mean_absolute_error','r2'), n_jobs=1, return_train_score=False)
    pipe.fit(X_train, y_train); pred = pipe.predict(X_test)
    rows.append({'Model_Type':'Ensemble','Model':name,'CV_RMSE':-scores['test_neg_root_mean_squared_error'].mean(),'CV_MAE':-scores['test_neg_mean_absolute_error'].mean(),'CV_R2':scores['test_r2'].mean(),'Test_RMSE':sqrt(mean_squared_error(y_test,pred)),'Test_MAE':mean_absolute_error(y_test,pred),'Test_R2':r2_score(y_test,pred)})
pd.DataFrame(rows).to_csv(REPORTS_DIR / 'final_results_summary.csv', index=False)

## Final summary
In the final task, I used Random Forest and Gradient Boosting. I also used the cluster labels from Task 3 as a new feature. These ensemble models are much better than the KNN from Task 2. Gradient Boosting is the winner because it has the lowest RMSE (~0.738). It is good at learning from its own mistakes. The clusters from Task 3 also helped to improve the score. This shows that mixing clustering and ensembles is the best way to work with this dataset.